# Day 5. Softmax / Probability

**오늘의 목표**
- Softmax가 "아무 숫자들"을 "확률"로 바꾸는 과정을 손으로 계산할 수 있다
- 왜 하필 `exp`(지수함수)를 쓰는지 안다
- `dim=-1`이 무엇을 뜻하는지, `(B, T, V)` 텐서에서 어느 축으로 Softmax를 하는지 안다
- Temperature와 Sampling으로 LLM이 다음 Token을 고르는 과정을 직접 해본다
- Attention에서 Softmax가 어떤 역할을 하는지 미리 본다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. 문제: 모델의 출력은 "확률"이 아니다

Day 3 연습 문제 Q4에서 만든 `lm_head`는 Token마다 **점수(logit)** 를 냅니다.

In [ ]:
vocab  = ["좋다", "춥다", "덥다", "맑다", "자동차"]
logits = torch.tensor([3.0, 2.0, 1.0, 0.5, -2.0])   # "오늘 날씨가" 다음 Token 점수

for w, s in zip(vocab, logits):
    print(f"{w:4s} {s.item():5.1f}")
print("합계:", logits.sum().item())

이 점수들은 그냥 "크면 좋다" 정도의 의미만 있습니다.
- 음수도 있고
- 합이 1도 아니라서
- "좋다가 나올 확률이 몇 %?"라고 물으면 답할 수 없습니다.

**확률이 되려면** 두 조건을 만족해야 합니다.
1. 모든 값이 **0 이상**
2. 전부 더하면 **1**

## 2. 첫 시도: 합으로 나누면 안 되나?

In [ ]:
naive = logits / logits.sum()
print("합으로 나누기:", r(naive, 3))
print("합계:", naive.sum().item())

합은 1이 됐지만 **자동차가 -0.444 (음수 확률!)** 입니다. 실패예요.

**해결책**: 먼저 모든 값을 **양수로 바꾼 뒤** 합으로 나눕니다.
양수로 바꾸는 데 쓰는 함수가 바로 **`exp` (지수함수, eˣ)** 입니다.

In [ ]:
x = torch.linspace(-3, 3, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(x, torch.exp(x), linewidth=2.5)
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.title("exp(x): 어떤 입력이든 항상 양수, 큰 값은 더 크게", fontsize=12)
plt.grid(alpha=0.3)
plt.show()

`exp`의 성질:
- 입력이 음수여도 결과는 **항상 양수** (`exp(-2) = 0.135`)
- 입력이 클수록 결과가 **훨씬 빠르게** 커짐 → 점수 차이를 더 벌려줌
- **순서는 그대로** 유지 (큰 점수는 여전히 큼)

## 3. Softmax 공식

```text
softmax(xᵢ) = exp(xᵢ) / Σ exp(xⱼ)
```

말로 하면: **"각 값에 exp를 씌우고, 전체 합으로 나눈다"**

### 3-1. 단계별로 직접 계산하기 (커리큘럼 예시)

In [ ]:
x = torch.tensor([2.0, 1.0, 0.1])

step1 = torch.exp(x)
step2 = step1.sum()
step3 = step1 / step2

print("1) exp(x)          :", r(step1, 3))
print("2) 합계             :", round(step2.item(), 3))
print("3) exp(x) / 합계    :", r(step3, 3))
print("   F.softmax 결과   :", r(F.softmax(x, dim=-1), 3))
print("   전체 합          :", round(step3.sum().item(), 4))

커리큘럼에 나온 `[2.0, 1.0, 0.1] → [0.66, 0.24, 0.10]`이 정확히 나왔죠?

### 3-2. 날씨 예시에 적용하기

In [ ]:
probs = F.softmax(logits, dim=-1)

for w, s, p in zip(vocab, logits, probs):
    bar = "█" * int(p.item() * 50)
    print(f"{w:4s}  점수 {s.item():5.1f}  →  확률 {p.item()*100:5.1f}%  {bar}")
print("합계:", round(probs.sum().item(), 4))

## 4. Softmax의 성질

### 4-1. 모든 값에 같은 수를 더해도 결과가 같다

In [ ]:
x = torch.tensor([2.0, 1.0, 0.1])
print("softmax(x)       :", r(F.softmax(x, dim=-1), 4))
print("softmax(x + 100) :", r(F.softmax(x + 100, dim=-1), 4))
print("softmax(x - 5)   :", r(F.softmax(x - 5, dim=-1), 4))

Softmax는 **점수의 절대값이 아니라 점수 사이의 차이**만 봅니다.
(`exp(a + c) / exp(b + c) = exp(a) / exp(b)` 이기 때문)

### 4-2. 그래서 생기는 문제와 해결: 숫자가 너무 크면?

In [ ]:
big = torch.tensor([1000.0, 999.0, 998.0])

naive = torch.exp(big) / torch.exp(big).sum()
print("exp(1000)        :", torch.exp(torch.tensor(1000.0)))
print("직접 계산 (그대로) :", naive)

stable = torch.exp(big - big.max()) / torch.exp(big - big.max()).sum()
print("최댓값 빼고 계산   :", stable)
print("F.softmax        :", F.softmax(big, dim=-1))

`exp(1000)`은 컴퓨터가 표현할 수 있는 범위를 넘어서 `inf`가 되고, `inf / inf = nan`이 됩니다.

4-1의 성질을 이용해 **최댓값을 먼저 빼고** 계산하면 해결됩니다. (가장 큰 값이 0이 되니까 `exp`가 1을 넘지 않음)
`F.softmax`는 내부에서 이 처리를 자동으로 해줘요. 연습 문제 Q2에서 직접 구현해봅니다.

## 5. `dim=-1`: 어느 축으로 Softmax를 하나?

텐서가 2차원 이상이면 **어느 방향으로 더해서 1을 만들지** 정해야 합니다.

GPT의 출력 shape는 `(B, T, V)`입니다.
- `B`: 문장 개수
- `T`: Token 위치
- `V`: Vocabulary 크기 (각 단어의 점수)

"각 위치에서 다음 Token의 확률"을 구해야 하니까 **V 축(마지막 축, `dim=-1`)** 으로 Softmax를 합니다.

In [ ]:
B, T, V = 2, 3, 5
logits_3d = torch.randn(B, T, V)

probs_right = F.softmax(logits_3d, dim=-1)   # 올바른 방법
probs_wrong = F.softmax(logits_3d, dim=1)    # 잘못된 방법 (T 축)

print("shape:", tuple(probs_right.shape), " ← Softmax는 shape를 안 바꿈")
print()
print("dim=-1 → 각 위치에서 V개 확률의 합:")
print(probs_right.sum(dim=-1))
print()
print("dim=1  → 각 위치에서 V개 확률의 합 (1이 아님!):")
print(probs_wrong.sum(dim=-1))

> 💡 `dim`을 잘못 주면 **에러 없이 틀린 결과**가 나옵니다. 그래서 더 위험해요.
> 확인 방법: `probs.sum(dim=-1)`을 찍어서 전부 1인지 보기.

## 6. LLM과의 연결 ①: 다음 Token 고르기

확률을 구했으니 이제 Token을 하나 골라야 합니다. 두 가지 방법이 있어요.

### 6-1. Greedy: 항상 확률이 제일 높은 것

In [ ]:
probs = F.softmax(logits, dim=-1)
idx = torch.argmax(probs).item()
print("Greedy 선택:", vocab[idx], "(10번 해도 항상 같음)")

### 6-2. Sampling: 확률대로 뽑기

`torch.multinomial`은 확률에 비례해서 랜덤으로 뽑습니다. "좋다"가 63%면 100번 중 63번쯤 나와요.
ChatGPT에 같은 질문을 해도 답이 매번 조금씩 다른 이유가 이거예요.

In [ ]:
samples = torch.multinomial(probs, num_samples=1000, replacement=True)
counts = torch.bincount(samples, minlength=len(vocab))

for w, p, c in zip(vocab, probs, counts):
    print(f"{w:4s}  확률 {p.item()*100:5.1f}%   1000번 중 {c.item():4d}번")

## 7. Temperature: 얼마나 과감하게 고를까?

Softmax 하기 전에 logits를 **Temperature(T)로 나눕니다**.

```text
probs = softmax(logits / T)
```

- `T < 1` (예: 0.5): 점수 차이가 **커짐** → 1등에 몰빵 → 안전하고 뻔한 답
- `T = 1`: 원래 그대로
- `T > 1` (예: 2.0): 점수 차이가 **줄어듦** → 고르게 퍼짐 → 다양하지만 엉뚱한 답

API에서 `temperature` 옵션을 본 적 있다면, 바로 이 숫자입니다.

In [ ]:
temps = [0.3, 1.0, 3.0]
fig, axes = plt.subplots(1, len(temps), figsize=(14, 3.5), sharey=True)
for ax, t in zip(axes, temps):
    p = F.softmax(logits / t, dim=-1)
    ax.bar(vocab, p, color="#4C72B0")
    for i, v in enumerate(p):
        ax.text(i, v.item() + 0.02, f"{v.item()*100:.0f}%", ha="center", fontsize=10)
    ax.set_title(f"Temperature = {t}", fontsize=13)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("확률")
plt.tight_layout()
plt.show()

## 8. LLM과의 연결 ②: Attention에서의 Softmax (미리보기)

Day 2에서 `Q @ K.T`로 Token 사이 관련도 점수 표 `(T, T)`를 만들었죠.
이 점수에 **행마다 Softmax**를 하면 "각 Token이 다른 Token을 몇 %씩 참고할지"가 됩니다.

In [ ]:
tokens = ["나는", "오늘", "학교에", "간다"]
torch.manual_seed(1)
scores = torch.randn(4, 4) * 2           # 가짜 Q @ K.T

weights = F.softmax(scores, dim=-1)       # 행마다 합이 1

print("Attention Weight (행 = 지금 Token, 열 = 참고하는 Token)")
print("        " + "".join(f"{t:>8s}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>6s}  " + "".join(f"{v.item()*100:7.1f}%" for v in row))

### 8-1. 미래를 못 보게 하기: `-inf` 트릭

GPT는 미래 Token을 보면 안 됩니다. (커리큘럼 3-5 Causal Mask, Day 14)
점수를 `-inf`로 바꾸면 `exp(-inf) = 0`이라서 **Softmax 후 확률이 정확히 0**이 됩니다.

In [ ]:
mask = torch.tril(torch.ones(4, 4))       # 아래 삼각형만 1
print("mask:")
print(mask)

masked_scores = scores.masked_fill(mask == 0, float("-inf"))
masked_weights = F.softmax(masked_scores, dim=-1)

print()
print("        " + "".join(f"{t:>8s}" for t in tokens))
for t, row in zip(tokens, masked_weights):
    print(f"{t:>6s}  " + "".join(f"{v.item()*100:7.1f}%" for v in row))

"나는"은 자기 자신만, "오늘"은 "나는·오늘"만 볼 수 있게 됐습니다. 각 행의 합은 여전히 100%예요.
이 트릭이 2주차 Causal Mask의 전부입니다.

## 9. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `x = [1.0, 1.0, 1.0, 1.0]`에 Softmax를 하면 어떻게 될까요? 실행하기 **전에** 예측하세요.

In [ ]:
x = torch.tensor([1.0, 1.0, 1.0, 1.0])

my_answer = "?"   # TODO: 리스트로

assert torch.allclose(torch.tensor(my_answer), F.softmax(x, dim=-1)), "모든 점수가 같다면 확률은?"
print("Q1 정답!")

**Q2.** `F.softmax`를 쓰지 않고 **숫자가 커도 터지지 않는** Softmax를 직접 구현하세요.
- 마지막 축(`dim=-1`) 기준으로 계산
- 힌트: `x.max(dim=-1, keepdim=True).values`, `torch.exp`, `.sum(dim=-1, keepdim=True)`

In [ ]:
def my_softmax(x):
    return None  # TODO

x = torch.randn(2, 3, 5)
assert torch.allclose(my_softmax(x), F.softmax(x, dim=-1), atol=1e-6)

big = torch.tensor([1000.0, 999.0, 998.0])
assert not torch.isnan(my_softmax(big)).any(), "큰 숫자에서 nan이 나와요. 4-2를 참고하세요"
print("Q2 정답!")

**Q3.** 아래 logits에 Temperature를 적용합니다. `T = 0.1`일 때와 `T = 100`일 때 "좋다"의 확률은 대략 어떻게 될까요?
가장 가까운 것을 고르세요: `"거의 0%"`, `"약 20%"`, `"거의 100%"`

In [ ]:
p_low  = F.softmax(logits / 0.1, dim=-1)[0].item()
p_high = F.softmax(logits / 100, dim=-1)[0].item()

answer_low  = "?"   # TODO: T = 0.1
answer_high = "?"   # TODO: T = 100 (힌트: 후보가 5개예요)

def closest(p):
    return min(["거의 0%", "약 20%", "거의 100%"], key=lambda k: abs({"거의 0%": 0, "약 20%": 0.2, "거의 100%": 1}[k] - p))

assert answer_low == closest(p_low), "T가 작으면 점수 차이가 어떻게 되나요?"
assert answer_high == closest(p_high), "T가 아주 크면 점수 차이가 어떻게 되나요?"
print(f"Q3 정답! T=0.1 → {p_low*100:.1f}%,  T=100 → {p_high*100:.1f}%")

**Q4.** 아래 Attention 점수에서 **3번째 Token(인덱스 2)은 아무도 보지 못하게** 가리고 Softmax 하세요.
- 3번째 **열**의 점수를 `-inf`로 바꾸면 됩니다
- 결과에서 3번째 열은 전부 0, 각 행의 합은 1이어야 해요

In [ ]:
scores = torch.randn(4, 4)

masked = scores.clone()
# TODO: masked의 3번째 열을 -inf로

weights = F.softmax(masked, dim=-1)
assert torch.all(weights[:, 2] == 0), "3번째 열이 0이 아니에요"
assert torch.allclose(weights.sum(dim=-1), torch.ones(4)), "각 행의 합이 1이 아니에요"
print("Q4 정답!")
print(weights.round(decimals=3))

## 10. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Softmax는 무엇을 무엇으로 바꾸는가? 결과는 어떤 두 조건을 만족하는가?
2. 왜 그냥 합으로 나누지 않고 `exp`를 먼저 씌우는가?
3. GPT 출력 `(B, T, V)`에 Softmax를 할 때 왜 `dim=-1`인가?
4. Temperature를 낮추면/높이면 생성되는 문장이 어떻게 달라지는가?
5. Attention에서 미래 Token의 점수를 `-inf`로 바꾸면 Softmax 결과가 어떻게 되는가?

**내 답:**

1.
2.
3.
4.
5.